# Loading The Pdf from Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Installing Necessary Libraries

In [ ]:
# Install PyPDF2 for PDF text extraction
!pip install PyPDF2

In [ ]:
# Install transformers for using Hugging Face models
!pip install transformers

In [ ]:
# Optional: Install Gradio if you want to test the chatbot in Colab itself
!pip install gradio

# Extracting Text from Pdf & Cleaning it

In [ ]:
# Import necessary libraries
import PyPDF2
import os
# Define the path to your PDF file
pdf_path = "/content/drive/MyDrive/fibromyalgia-information-booklet-july2021.pdf"


In [ ]:
# Initialize an empty string to store the text
full_text = ""

# Open and read the PDF file
with open(pdf_path, "rb") as file:
    # Create a PDF reader object
    pdf_reader = PyPDF2.PdfReader(file)

    # Iterate through all pages and extract text
    for page_num in range(len(pdf_reader.pages)):
        page = pdf_reader.pages[page_num]
        full_text += page.extract_text()

# Print the entire extracted text
print(full_text)

Fibromyalgia
Words shown in bold are explained in the glossary on p.29.We’re the 10 million people living with arthritis. We’re the carers, 
researchers, health professionals, friends and parents all united in  our ambition to ensure that one day, no one will have to live with  the pain, fatigue and isolation that arthritis causes.
We understand that every day is different. We know that what works 
for one person may not help someone else. Our information is a collaboration of experiences, research and facts. We aim to give you everything you need to know about your condition, the treatments available and the many options you can try, so you can make the best and most informed choices for your lifestyle.
We’re always happy to hear from you whether it’s with feedback on our 
information, to share your story, or just to find out more about the work of Versus Arthritis. Contact us at content@versusarthritis.org
Registered office: Versus Arthritis, Copeman House, St Mary’s Gate, Chesterfie

# Defining The model distilbert-base-cased-distilled-squad

In [ ]:
# Import necessary libraries
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline

# Define the model and tokenizer names
# Using DistilBERT which is lighter and faster while maintaining good performance
model_name = "distilbert-base-cased-distilled-squad"

# Download and load the tokenizer and model
print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

# Create a question-answering pipeline
qa_pipeline = pipeline(
    'question-answering',
    model=model,
    tokenizer=tokenizer
)

# Test the model with a sample question
def get_answer(question, context):
    result = qa_pipeline({
        'question': question,
        'context': context
    })
    return result

# Example usage
sample_question = "What is fibromyalgia?"
answer = get_answer(sample_question, full_text)
print("\nQuestion:", sample_question)
print("Answer:", answer['answer'])
print("Confidence Score:", answer['score'])

# Function to handle longer texts by splitting into chunks
def get_answer_from_long_text(question, context, max_length=512):
    # Split the context into chunks
    words = context.split()
    chunks = []
    current_chunk = []
    current_length = 0

    for word in words:
        current_chunk.append(word)
        current_length += len(word) + 1  # +1 for space

        if current_length >= max_length:
            chunks.append(' '.join(current_chunk))
            current_chunk = []
            current_length = 0

    if current_chunk:
        chunks.append(' '.join(current_chunk))

    # Get answers from all chunks
    answers = []
    for chunk in chunks:
        result = qa_pipeline({
            'question': question,
            'context': chunk
        })
        answers.append(result)

    # Return the answer with the highest confidence score
    best_answer = max(answers, key=lambda x: x['score'])
    return best_answer

# Test the long text function
test_question = "What are the main symptoms of fibromyalgia?"
answer = get_answer_from_long_text(test_question, full_text)
print("\nQuestion:", test_question)
print("Answer:", answer['answer'])
print("Confidence Score:", answer['score'])

Loading tokenizer and model...

Question: What is fibromyalgia?
Answer: varies from person to person
Confidence Score: 0.9321709275245667

Question: What are the main symptoms of fibromyalgia?
Answer: Stress and unhappiness
Confidence Score: 0.9951866269111633


# Question Answering Pipeline

In [ ]:
# Import necessary libraries
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline
import textwrap

class FibromyalgiaQASystem:
    def __init__(self, model_name="distilbert-base-cased-distilled-squad"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForQuestionAnswering.from_pretrained(model_name)
        self.qa_pipeline = pipeline(
            'question-answering',
            model=self.model,
            tokenizer=self.tokenizer
        )
        self.context = ""

    def set_context(self, text):
        """Set the context for question answering"""
        self.context = text

    def get_answer(self, question, max_length=512):
        """Get answer for a question using the stored context"""
        # Split context into manageable chunks if it's too long
        chunks = textwrap.wrap(self.context, max_length)

        answers = []
        for chunk in chunks:
            try:
                result = self.qa_pipeline({
                    'question': question,
                    'context': chunk
                })
                answers.append(result)
            except Exception as e:
                print(f"Error processing chunk: {e}")
                continue

        if not answers:
            return {
                'answer': 'I could not find an answer in the provided text.',
                'score': 0.0
            }

        # Get the answer with highest confidence score
        best_answer = max(answers, key=lambda x: x['score'])

        # Format the response
        response = {
            'question': question,
            'answer': best_answer['answer'],
            'confidence': round(best_answer['score'], 4),
            'source': 'Fibromyalgia Information Booklet'
        }

        return response

# Initialize the QA system
qa_system = FibromyalgiaQASystem()

# Set the context (using the previously extracted text)
qa_system.set_context(full_text)

# Function to format and print answers
def print_answer(response):
    print("\nQ:", response['question'])
    print("A:", response['answer'])
    print(f"Confidence: {response['confidence']}")
    print(f"Source: {response['source']}")
    print("-" * 50)

# Test the system with some sample questions
sample_questions = [
    "What is fibromyalgia?",
    "What are the main symptoms of fibromyalgia?",
    "How is fibromyalgia diagnosed?",
    "What treatments are available for fibromyalgia?",
    "Can exercise help with fibromyalgia?",
]

# Test the pipeline with sample questions
def test_pipeline():
    print("Testing QA Pipeline with sample questions...")
    print("=" * 50)

    for question in sample_questions:
        response = qa_system.get_answer(question)
        print_answer(response)

# Additional utility functions for specific types of questions
def get_treatment_info(condition):
    """Get specific treatment information"""
    question = f"What treatments are available for {condition}?"
    return qa_system.get_answer(question)

def get_symptom_info(condition):
    """Get specific symptom information"""
    question = f"What are the symptoms of {condition}?"
    return qa_system.get_answer(question)

def get_management_tips():
    """Get self-management tips"""
    question = "How can patients help themselves manage fibromyalgia?"
    return qa_system.get_answer(question)

# Run the test
if __name__ == "__main__":
    print("Initializing Fibromyalgia QA System...")
    test_pipeline()

    # Example of using specific question functions
    print("\nGetting specific treatment information:")
    treatment_info = get_treatment_info("fibromyalgia")
    print_answer(treatment_info)

    print("\nGetting self-management tips:")
    management_tips = get_management_tips()
    print_answer(management_tips)

Initializing Fibromyalgia QA System...
Testing QA Pipeline with sample questions...

Q: What is fibromyalgia?
A: keeping active
Confidence: 0.9893
Source: Fibromyalgia Information Booklet
--------------------------------------------------

Q: What are the main symptoms of fibromyalgia?
A: Stress and unhappiness
Confidence: 0.9956
Source: Fibromyalgia Information Booklet
--------------------------------------------------

Q: How is fibromyalgia diagnosed?
A: Poor sleep
Confidence: 0.9318
Source: Fibromyalgia Information Booklet
--------------------------------------------------

Q: What treatments are available for fibromyalgia?
A: Stress and unhappiness
Confidence: 0.9912
Source: Fibromyalgia Information Booklet
--------------------------------------------------

Q: Can exercise help with fibromyalgia?
A: Massage
Confidence: 0.7325
Source: Fibromyalgia Information Booklet
--------------------------------------------------

Getting specific treatment information:

Q: What treatments are

# Gradio Chatbot Interface & Deployment

In [ ]:
# Install required packages if not already installed
!pip install -q gradio
!pip install -q transformers
!pip install -q torch

# Import necessary libraries
import gradio as gr
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline

# Load the model and create QA pipeline (reusing previous code)
class FibromyalgiaQASystem:
    def __init__(self, model_name="distilbert-base-cased-distilled-squad"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForQuestionAnswering.from_pretrained(model_name)
        self.qa_pipeline = pipeline(
            'question-answering',
            model=self.model,
            tokenizer=self.tokenizer
        )
        self.context = ""

    def set_context(self, text):
        self.context = text

    def get_answer(self, question):
        if not question:
            return "Please ask a question."

        try:
            result = self.qa_pipeline({
                'question': question,
                'context': self.context
            })

            response = f"""Answer: {result['answer']}
Confidence Score: {result['score']:.2%}"""

            return response
        except Exception as e:
            return f"An error occurred: {str(e)}"

# Initialize the QA system with the extracted text
qa_system = FibromyalgiaQASystem()
qa_system.set_context(full_text)  # full_text is your previously extracted PDF content

# Create the Gradio interface
def answer_question(question):
    return qa_system.get_answer(question)

# Define the Gradio interface
iface = gr.Interface(
    fn=answer_question,
    inputs=[
        gr.Textbox(label="Ask a question about Fibromyalgia",
                  placeholder="Example: What is fibromyalgia?",
                  lines=2)
    ],
    outputs=gr.Textbox(label="Answer", lines=4),
    title="Fibromyalgia Q&A Assistant",
    description="Ask questions about Fibromyalgia and get answers from the medical document.",
    examples=[
        ["What is fibromyalgia?"],
        ["What are the main symptoms of fibromyalgia?"],
        ["How is fibromyalgia diagnosed?"],
        ["What treatments are available?"],
        ["Does fibromyalgia run in families?"]
    ],
    theme=gr.themes.Soft()
)

# Launch the interface
iface.launch(share=True, debug=True)

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Running on public URL: https://02bd0c561c1e31c6fd.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://02bd0c561c1e31c6fd.gradio.live


In [ ]:
# Install required packages
!pip install -q gradio
!pip install -q transformers
!pip install -q torch
!pip install -q PyPDF2

# Import necessary libraries
import gradio as gr
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline
import PyPDF2
import torch

# PDF text extraction
def extract_text_from_pdf(pdf_path):
    full_text = ""
    with open(pdf_path, "rb") as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page_num in range(len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            full_text += page.extract_text()
    return full_text

# Extract text from PDF
pdf_path = "/content/drive/MyDrive/fibromyalgia-information-booklet-july2021.pdf"
document_text = extract_text_from_pdf(pdf_path)

# QA Pipeline Class
class FibromyalgiaQASystem:
    def __init__(self, context, model_name="distilbert-base-cased-distilled-squad"):
        print("Initializing QA System...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForQuestionAnswering.from_pretrained(model_name)
        self.qa_pipeline = pipeline(
            'question-answering',
            model=self.model,
            tokenizer=self.tokenizer,
            device=0 if torch.cuda.is_available() else -1
        )
        self.context = context
        print("QA System initialized!")

    def answer_question(self, question):
        if not question.strip():
            return "Please ask a question."

        try:
            # Process the question
            result = self.qa_pipeline({
                'question': question,
                'context': self.context
            })

            # Format the response
            response = f"""
Answer: {result['answer']}

Confidence Score: {result['score']:.2%}

Note: This answer is based on the provided medical document. Please consult healthcare professionals for medical advice.
            """
            return response

        except Exception as e:
            return f"An error occurred: {str(e)}"

# Initialize QA System
qa_system = FibromyalgiaQASystem(document_text)

# Gradio Interface
def create_gradio_interface():
    # CSS for custom styling
    custom_css = """
    #question_box { height: 100px; }
    #answer_box { height: 200px; }
    """

    # Create the interface
    iface = gr.Interface(
        fn=qa_system.answer_question,
        inputs=[
            gr.Textbox(
                label="Your Question",
                placeholder="Type your question about fibromyalgia here...",
                elem_id="question_box"
            )
        ],
        outputs=[
            gr.Textbox(
                label="Answer",
                elem_id="answer_box"
            )
        ],
        title="Fibromyalgia Medical Assistant",
        description="""
        This AI assistant can answer your questions about fibromyalgia based on medical documentation.
        Ask any question about symptoms, diagnosis, treatment, or management of fibromyalgia.
        """,
        examples=[
            ["What is fibromyalgia?"],
            ["What are the main symptoms of fibromyalgia?"],
            ["How is fibromyalgia diagnosed?"],
            ["What treatments are available for fibromyalgia?"],
            ["Does fibromyalgia run in families?"],
            ["Can exercise help with fibromyalgia?"]
        ],
        theme=gr.themes.Soft(),
        css=custom_css
    )
    return iface

# Create and launch the interface
demo = create_gradio_interface()
demo.launch(debug=True, share=True)

Initializing QA System...


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


QA System initialized!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Running on public URL: https://fff09db5168317579b.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://fff09db5168317579b.gradio.live


In [ ]:
# Install required packages
!pip install -q gradio transformers torch PyPDF2

# Import necessary libraries
import gradio as gr
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline
import PyPDF2
import torch

# Function to extract text from a PDF file
def extract_text_from_pdf(pdf_path):
    full_text = ""
    with open(pdf_path, "rb") as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page_num in range(len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            full_text += page.extract_text()
    return full_text

# Path to the PDF file stored in Google Drive
pdf_path = "/content/drive/MyDrive/fibromyalgia-information-booklet-july2021.pdf"
# Extract the text from the PDF document
document_text = extract_text_from_pdf(pdf_path)

# Define the QA system class
class FibromyalgiaQASystem:
    def __init__(self, context, model_name="distilbert-base-cased-distilled-squad"):
        print("Initializing QA System...")
        # Load the model and tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForQuestionAnswering.from_pretrained(model_name)
        # Create a question-answering pipeline using the model
        self.qa_pipeline = pipeline(
            'question-answering',
            model=self.model,
            tokenizer=self.tokenizer,
            device=0 if torch.cuda.is_available() else -1  # Use GPU if available
        )
        self.context = context
        print("QA System initialized!")

    def answer_question(self, question):
        if not question.strip():
            return "Please ask a question."

        try:
            # Generate answer for the question based on the context
            result = self.qa_pipeline({
                'question': question,
                'context': self.context
            })
            # Format the response
            response = f"""
Answer: {result['answer']}

Confidence Score: {result['score']:.2%}

Note: This answer is based on the provided medical document. Please consult healthcare professionals for medical advice.
            """
            return response
        except Exception as e:
            return f"An error occurred: {str(e)}"

# Initialize the QA system with extracted PDF text
qa_system = FibromyalgiaQASystem(document_text)

# Create the Gradio interface
def create_gradio_interface():
    # Custom CSS styling for the interface
    custom_css = """
    #question_box { height: 100px; }
    #answer_box { height: 200px; }
    """

    # Create the Gradio interface with custom inputs and outputs
    iface = gr.Interface(
        fn=qa_system.answer_question,
        inputs=[
            gr.Textbox(
                label="Your Question",
                placeholder="Type your question about fibromyalgia here...",
                elem_id="question_box"
            )
        ],
        outputs=[
            gr.Textbox(
                label="Answer",
                elem_id="answer_box"
            )
        ],
        title="Fibromyalgia Medical Assistant",
        description="""
        This AI assistant can answer your questions about fibromyalgia based on medical documentation.
        Ask any question about symptoms, diagnosis, treatment, or management of fibromyalgia.
        """,
        examples=[
            ["What is fibromyalgia?"],
            ["What are the main symptoms of fibromyalgia?"],
            ["How is fibromyalgia diagnosed?"],
            ["What treatments are available for fibromyalgia?"],
            ["Does fibromyalgia run in families?"],
            ["Can exercise help with fibromyalgia?"]
        ],
        theme=gr.themes.Soft(),
        css=custom_css
    )
    return iface

# Create and launch the Gradio interface
demo = create_gradio_interface()
demo.launch(debug=True, share=True)


Initializing QA System...


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


QA System initialized!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Running on public URL: https://4a63d690cdea5e57aa.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://4a63d690cdea5e57aa.gradio.live


# Adding More Question Answers Pipeline

In [ ]:
# Import necessary libraries
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline
import textwrap

class FibromyalgiaQASystem:
    def __init__(self, model_name="distilbert-base-cased-distilled-squad"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForQuestionAnswering.from_pretrained(model_name)
        self.qa_pipeline = pipeline(
            'question-answering',
            model=self.model,
            tokenizer=self.tokenizer
        )
        self.context = ""

    def set_context(self, text):
        """Set the context for question answering"""
        self.context = text

    def get_answer(self, question, max_length=512):
        """Get answer for a question using the stored context"""
        # Split context into manageable chunks if it's too long
        chunks = textwrap.wrap(self.context, max_length)

        answers = []
        for chunk in chunks:
            try:
                result = self.qa_pipeline({
                    'question': question,
                    'context': chunk
                })
                answers.append(result)
            except Exception as e:
                print(f"Error processing chunk: {e}")
                continue

        if not answers:
            return {
                'answer': 'I could not find an answer in the provided text.',
                'score': 0.0
            }

        # Get the answer with highest confidence score
        best_answer = max(answers, key=lambda x: x['score'])

        # Format the response
        response = {
            'question': question,
            'answer': best_answer['answer'],
            'confidence': round(best_answer['score'], 4),
            'source': 'Fibromyalgia Information Booklet'
        }

        return response

# Initialize the QA system
qa_system = FibromyalgiaQASystem()

# Set the context (using the previously extracted text)
qa_system.set_context(full_text)

# Function to format and print answers
def print_answer(response):
    print("\nQ:", response['question'])
    print("A:", response['answer'])
    print(f"Confidence: {response['confidence']}")
    print(f"Source: {response['source']}")
    print("-" * 50)

# Test the system with some sample questions
sample_questions = [
    "What is fibromyalgia?",
    "What are the main symptoms of fibromyalgia?",
    "How is fibromyalgia diagnosed?",
    "What treatments are available for fibromyalgia?",
    "Can exercise help with fibromyalgia?",
    "Are there any dietary recommendations for fibromyalgia?",
    "What lifestyle changes can help manage fibromyalgia symptoms?",
    "What causes fibromyalgia?",
    "Is fibromyalgia a progressive disease?",
    "How common is fibromyalgia among different demographics?",
    "Are there support groups for people with fibromyalgia?",
    "What is the difference between fibromyalgia and chronic fatigue syndrome?",
]

# Test the pipeline with sample questions
def test_pipeline():
    print("Testing QA Pipeline with sample questions...")
    print("=" * 50)

    for question in sample_questions:
        response = qa_system.get_answer(question)
        print_answer(response)

# Additional utility functions for specific types of questions
def get_treatment_info(condition):
    """Get specific treatment information"""
    question = f"What treatments are available for {condition}?"
    return qa_system.get_answer(question)

def get_symptom_info(condition):
    """Get specific symptom information"""
    question = f"What are the symptoms of {condition}?"
    return qa_system.get_answer(question)

def get_management_tips():
    """Get self-management tips"""
    question = "How can patients help themselves manage fibromyalgia?"
    return qa_system.get_answer(question)

# Run the test
if __name__ == "__main__":
    print("Initializing Fibromyalgia QA System...")
    test_pipeline()

    # Example of using specific question functions
    print("\nGetting specific treatment information:")
    treatment_info = get_treatment_info("fibromyalgia")
    print_answer(treatment_info)

    print("\nGetting self-management tips:")
    management_tips = get_management_tips()
    print_answer(management_tips)


Initializing Fibromyalgia QA System...
Testing QA Pipeline with sample questions...

Q: What is fibromyalgia?
A: keeping active
Confidence: 0.9893
Source: Fibromyalgia Information Booklet
--------------------------------------------------

Q: What are the main symptoms of fibromyalgia?
A: Stress and unhappiness
Confidence: 0.9956
Source: Fibromyalgia Information Booklet
--------------------------------------------------

Q: How is fibromyalgia diagnosed?
A: Poor sleep
Confidence: 0.9318
Source: Fibromyalgia Information Booklet
--------------------------------------------------

Q: What treatments are available for fibromyalgia?
A: Stress and unhappiness
Confidence: 0.9912
Source: Fibromyalgia Information Booklet
--------------------------------------------------

Q: Can exercise help with fibromyalgia?
A: Massage
Confidence: 0.7325
Source: Fibromyalgia Information Booklet
--------------------------------------------------

Q: Are there any dietary recommendations for fibromyalgia?
A: C

# Chatbot Interface

In [ ]:
# Import necessary libraries
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline
import PyPDF2
import textwrap
import gradio as gr
import torch

# Function to extract text from a PDF file
def extract_text_from_pdf(pdf_path):
    full_text = ""
    with open(pdf_path, "rb") as file:
        pdf_reader = PyPDF2.PdfReader(file)
        for page_num in range(len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            full_text += page.extract_text()
    return full_text

# Path to the PDF file stored in Google Drive
pdf_path = "/content/drive/MyDrive/fibromyalgia-information-booklet-july2021.pdf"
# Extract the text from the PDF document
document_text = extract_text_from_pdf(pdf_path)

# Define the QA system class
class FibromyalgiaQASystem:
    def __init__(self, context, model_name="distilbert-base-cased-distilled-squad"):
        print("Initializing QA System...")
        # Load the model and tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForQuestionAnswering.from_pretrained(model_name)
        # Create a question-answering pipeline using the model
        self.qa_pipeline = pipeline(
            'question-answering',
            model=self.model,
            tokenizer=self.tokenizer,
            device=0 if torch.cuda.is_available() else -1  # Use GPU if available
        )
        self.context = context
        print("QA System initialized!")

    def answer_question(self, question):
        if not question.strip():
            return "Please ask a question."

        # Split context into manageable chunks if it's too long
        chunks = textwrap.wrap(self.context, 512)

        answers = []
        for chunk in chunks:
            try:
                result = self.qa_pipeline({
                    'question': question,
                    'context': chunk
                })
                answers.append(result)
            except Exception as e:
                print(f"Error processing chunk: {e}")
                continue

        if not answers:
            return "I could not find an answer in the provided text."

        # Get the answer with highest confidence score
        best_answer = max(answers, key=lambda x: x['score'])

        # Format the response
        response = f"""
Answer: {best_answer['answer']}

Confidence Score: {best_answer['score']:.2%}

Note: This answer is based on the provided medical document. Please consult healthcare professionals for medical advice.
        """
        return response

# Initialize the QA system with extracted PDF text
qa_system = FibromyalgiaQASystem(document_text)

# Create the Gradio interface
def create_gradio_interface():
    # Custom CSS styling for the interface
    custom_css = """
    #question_box { height: 100px; }
    #answer_box { height: 200px; }
    """

    # Create the Gradio interface with custom inputs and outputs
    iface = gr.Interface(
        fn=qa_system.answer_question,
        inputs=[
            gr.Textbox(
                label="Your Question",
                placeholder="Type your question about fibromyalgia here...",
                elem_id="question_box"
            )
        ],
        outputs=[
            gr.Textbox(
                label="Answer",
                elem_id="answer_box"
            )
        ],
        title="Fibromyalgia Medical Assistant",
        description="""
        This AI assistant can answer your questions about fibromyalgia based on medical documentation.
        Ask any question about symptoms, diagnosis, treatment, or management of fibromyalgia.
        """,
        examples=[
            ["What is fibromyalgia?"],
            ["What are the main symptoms of fibromyalgia?"],
            ["How is fibromyalgia diagnosed?"],
            ["What treatments are available for fibromyalgia?"],
            ["Can exercise help with fibromyalgia?"],
            ["Are there any dietary recommendations for fibromyalgia?"],
            ["What lifestyle changes can help manage fibromyalgia symptoms?"],
            ["What causes fibromyalgia?"],
            ["Is fibromyalgia a progressive disease?"],
            ["How common is fibromyalgia among different demographics?"],
            ["Are there support groups for people with fibromyalgia?"],
            ["What is the difference between fibromyalgia and chronic fatigue syndrome?"],
        ],
        theme=gr.themes.Soft(),
        css=custom_css
    )
    return iface

# Create and launch the Gradio interface
demo = create_gradio_interface()
demo.launch(debug=True, share=True)



Initializing QA System...
QA System initialized!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Running on public URL: https://5e83a7d25feb3f9c92.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://5e83a7d25feb3f9c92.gradio.live


#  Alternative version with a chat-like interface:

In [ ]:
import gradio as gr
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline
import time

class EnhancedFibromyalgiaQASystem:
    def __init__(self, model_name="distilbert-base-cased-distilled-squad"):
        print("Loading model and tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForQuestionAnswering.from_pretrained(model_name)
        self.qa_pipeline = pipeline(
            'question-answering',
            model=self.model,
            tokenizer=self.tokenizer
        )
        self.context = ""
        print("System initialized successfully!")

    def set_context(self, text):
        self.context = text

    def get_answer(self, question, history):
        if not question:
            return "", history

        try:
            # Add user question to history
            history.append(("You: " + question, None))

            # Get answer from model
            result = self.qa_pipeline({
                'question': question,
                'context': self.context
            })

            # Format response
            response = f"""Based on the medical document:

Answer: {result['answer']}

Confidence: {result['score']:.2%}"""

            # Add response to history
            history[-1] = (history[-1][0], response)

            return "", history
        except Exception as e:
            error_message = f"An error occurred: {str(e)}"
            history[-1] = (history[-1][0], error_message)
            return "", history

# Initialize the QA system
qa_system = EnhancedFibromyalgiaQASystem()
qa_system.set_context(full_text)

# Create the Gradio interface with chat-like interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🏥 Fibromyalgia Medical Assistant

    Ask questions about Fibromyalgia and get answers based on the medical document.

    **Example questions you can ask:**
    - What is fibromyalgia?
    - What are the main symptoms?
    - How is it diagnosed?
    - What treatments are available?
    """)

    chatbot = gr.Chatbot(label="Chat History")
    msg = gr.Textbox(label="Ask your question here", placeholder="Type your question and press Enter")
    clear = gr.Button("Clear")

    # Store conversation history
    state = gr.State([])

    # Handle user input
    msg.submit(
        qa_system.get_answer,
        [msg, state],
        [msg, chatbot]
    )

    # Clear conversation
    clear.click(lambda: [], None, chatbot)

    gr.Markdown("""
    ### ℹ️ About this Assistant
    - This assistant uses AI to answer questions about Fibromyalgia based on a medical document
    - Answers are provided with confidence scores
    - If you're experiencing symptoms, please consult with a healthcare professional
    """)

# Launch the interface
demo.launch(share=True, debug=True)

Loading model and tokenizer...


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


System initialized successfully!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Running on public URL: https://3a0b35eaa51a37ec8e.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3a0b35eaa51a37ec8e.gradio.live


In [ ]:
import gradio as gr
from transformers import AutoModelForQuestionAnswering, AutoTokenizer, pipeline
import time

class EnhancedFibromyalgiaQASystem:
    def __init__(self, model_name="distilbert-base-cased-distilled-squad"):
        print("Loading model and tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForQuestionAnswering.from_pretrained(model_name)
        self.qa_pipeline = pipeline(
            'question-answering',
            model=self.model,
            tokenizer=self.tokenizer
        )
        self.context = ""
        print("System initialized successfully!")

    def set_context(self, text):
        self.context = text

    def get_answer(self, question, history):
        if not question:
            return "", history

        try:
            # Add user question to history
            history.append(("You: " + question, None))

            # Get answer from model
            result = self.qa_pipeline({
                'question': question,
                'context': self.context
            })

            # Format response
            response = f"""Based on the medical document:

Answer: {result['answer']}

Confidence: {result['score']:.2%}"""

            # Add response to history
            history[-1] = (history[-1][0], response)

            return "", history
        except Exception as e:
            error_message = f"An error occurred: {str(e)}"
            history[-1] = (history[-1][0], error_message)
            return "", history

# Initialize the QA system
qa_system = EnhancedFibromyalgiaQASystem()
qa_system.set_context(full_text)

# Create the Gradio interface with chat-like interface
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🏥 Fibromyalgia Medical Assistant

    Ask questions about Fibromyalgia and get answers based on the medical document.

    **Example questions you can ask:**
    - What is fibromyalgia?
    - What are the main symptoms?
    - How is it diagnosed?
    - What treatments are available?
    """)

    chatbot = gr.Chatbot(label="Chat History")
    msg = gr.Textbox(label="Ask your question here", placeholder="Type your question and press Enter")
    clear = gr.Button("Clear")

    # Store conversation history
    state = gr.State([])

    # Handle user input
    msg.submit(
        qa_system.get_answer,
        [msg, state],
        [msg, chatbot]
    )

    # Clear conversation
    clear.click(lambda: [], None, chatbot)

    gr.Markdown("""
    ### ℹ️ About this Assistant
    - This assistant uses AI to answer questions about Fibromyalgia based on a medical document
    - Answers are provided with confidence scores
    - If you're experiencing symptoms, please consult with a healthcare professional
    """)

# Launch the interface
demo.launch(share=True, debug=True)

Loading model and tokenizer...


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


System initialized successfully!
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Running on public URL: https://89a6d74c3d7ee2884e.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://093b915645aa60bcd5.gradio.live
Killing tunnel 127.0.0.1:7860 <> https://89a6d74c3d7ee2884e.gradio.live
